In [1]:
# Step 1: Import all necessary libraries
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Download NLTK stopwords (common words like 'the', 'and' that add little value)
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [2]:
# Step 2: Load the movie_reviews dataset from NLTK
nltk.download('movie_reviews', quiet=True)
from nltk.corpus import movie_reviews

# Extract file IDs and their corresponding categories (pos/neg)
documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

# Convert to a pandas DataFrame for easier handling
data = []
for words, sentiment in documents:
    # Join the list of words into a single string
    text = ' '.join(words)
    data.append([text, sentiment])

df = pd.DataFrame(data, columns=['review', 'sentiment'])
print(f"Total samples: {len(df)}")
print(df.head())

Total samples: 2000
                                              review sentiment
0  plot : two teen couples go to a church party ,...       neg
1  the happy bastard ' s quick movie review damn ...       neg
2  it is movies like these that make a jaded movi...       neg
3  " quest for camelot " is warner bros . ' first...       neg
4  synopsis : a mentally unstable man undergoing ...       neg


In [3]:
# Step 3: Text cleaning function
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # 1. Convert to lowercase
    text = text.lower()
    # 2. Remove punctuation and digits (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # 3. Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    # 4. Remove stopwords (optional but often helps)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

# Apply cleaning to all reviews
df['cleaned_review'] = df['review'].apply(clean_text)

print("Before cleaning:")
print(df['review'][0][:200])
print("\nAfter cleaning:")
print(df['cleaned_review'][0][:200])

Before cleaning:
plot : two teen couples go to a church party , drink and then drive . they get into an accident . one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . what

After cleaning:
plot two teen couples go church party drink drive get accident one guys dies girlfriend continues see life nightmares deal watch movie sorta find critique mind fuck movie teen generation touches cool 


In [4]:
# Step 4: Encode labels: 'pos' -> 1 (Positive), 'neg' -> 0 (Negative)
df['label'] = df['sentiment'].map({'pos': 1, 'neg': 0})

print(df[['sentiment', 'label']].head())

  sentiment  label
0       neg      0
1       neg      0
2       neg      0
3       neg      0
4       neg      0


In [5]:
# Step 5: Train-test split (80% train, 20% test)
X = df['cleaned_review']   # features (text)
y = df['label']            # target (0/1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 1600
Testing samples: 400


In [6]:
# Step 6: TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

# Fit on training data and transform both train and test
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Shape of training features: {X_train_tfidf.shape}")

Shape of training features: (1600, 5000)


In [7]:
# Step 7: Train the model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

print("Training completed!")

Training completed!


In [8]:
# Step 8: Predict on test set and evaluate
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8375

Classification Report:
              precision    recall  f1-score   support

    Negative       0.85      0.81      0.83       200
    Positive       0.82      0.86      0.84       200

    accuracy                           0.84       400
   macro avg       0.84      0.84      0.84       400
weighted avg       0.84      0.84      0.84       400


Confusion Matrix:
[[163  37]
 [ 28 172]]


In [9]:
# Step 9: Function to predict sentiment of a new text
def predict_sentiment(text):
    # 1. Clean the text
    cleaned = clean_text(text)
    # 2. Convert to TF-IDF using the same vectorizer
    text_tfidf = vectorizer.transform([cleaned])
    # 3. Predict
    pred = model.predict(text_tfidf)[0]
    # 4. Return label
    return "Positive" if pred == 1 else "Negative"

# Test with the examples from your image
examples = [
    "I love this product",
    "This is amazing",
    "I am very happy",
    "I hate this",
    "This is terrible",
    "I am disappointed"
]

print("\n--- Testing on your examples ---")
for ex in examples:
    print(f"Text: {ex:20} -> Sentiment: {predict_sentiment(ex)}")


--- Testing on your examples ---
Text: I love this product  -> Sentiment: Positive
Text: This is amazing      -> Sentiment: Positive
Text: I am very happy      -> Sentiment: Positive
Text: I hate this          -> Sentiment: Positive
Text: This is terrible     -> Sentiment: Negative
Text: I am disappointed    -> Sentiment: Positive


In [11]:
# Step 10: Interactive testing (run this cell multiple times)
custom = input("Enter a sentence: ")
print(f"Sentiment: {predict_sentiment(custom)}")

Enter a sentence: The movie was boring and too long
Sentiment: Negative


In [12]:
# Save model and vectorizer using joblib (better for large scikit-learn objects)
import joblib

# Save the trained model
joblib.dump(model, 'sentiment_model.pkl')

# Save the TF-IDF vectorizer
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("Model and vectorizer saved as .pkl files")

Model and vectorizer saved as .pkl files


In [13]:
from google.colab import files
files.download('sentiment_model.pkl')
files.download('tfidf_vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>